In [4]:
import pandas as pd

pd.set_option("display.max_columns", None)
caminho = "../data/raw/"

train = pd.read_csv(caminho + "train.csv", parse_dates=["date"])
stores = pd.read_csv(caminho + "stores.csv")
transactions = pd.read_csv(caminho + "transactions.csv", parse_dates=["date"])
oil = pd.read_csv(caminho + "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(caminho + "holidays_events.csv", parse_dates=["date"])

In [5]:
tabelas = {"train": train, "stores": stores, "transactions": transactions,
           "oil": oil, "holidays": holidays}

for nome, df in tabelas.items():
    print(f"{nome}: {df.shape[0]:,} linhas, {df.shape[1]} colunas")

train: 3,000,888 linhas, 6 colunas
stores: 54 linhas, 5 colunas
transactions: 83,488 linhas, 3 colunas
oil: 1,218 linhas, 2 colunas
holidays: 350 linhas, 6 colunas


In [6]:
for nome, df in tabelas.items():
    print(f"\n===== {nome} =====")
    display(df.head())
    


===== train =====


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0



===== stores =====


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4



===== transactions =====


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922



===== oil =====


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20



===== holidays =====


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


## Primeiras observações
- Cada linha do train = uma loja, uma família de produto, um dia.
- Lojas fechadas não aparecem no transactions (ex.: 01/01/2013).
- O oil não tem preços em fins de semana e possui valores vazios.
- Holidays começa em 2012, antes das vendas.
- Feriados transferidos precisam de atenção.

In [10]:
for nome, df in tabelas.items():
    print(f"\n{nome}:")
    print(df.isnull().sum())


train:
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

stores:
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

transactions:
date            0
store_nbr       0
transactions    0
dtype: int64

oil:
date           0
dcoilwtico    43
dtype: int64

holidays:
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64


In [11]:
print("Período:", train["date"].min(), "até", train["date"].max())
print("Lojas:", train["store_nbr"].nunique())
print("Famílias de produto:", train["family"].nunique())

Período: 2013-01-01 00:00:00 até 2017-08-15 00:00:00
Lojas: 54
Famílias de produto: 33


In [12]:
todas_datas = pd.date_range(train["date"].min(), train["date"].max())
datas_no_train = train["date"].unique()
faltando = todas_datas.difference(datas_no_train)

print(len(todas_datas), "dias no período")
print(len(datas_no_train), "dias no train")
print("Datas faltando:", list(faltando.date))

1688 dias no período
1684 dias no train
Datas faltando: [datetime.date(2013, 12, 25), datetime.date(2014, 12, 25), datetime.date(2015, 12, 25), datetime.date(2016, 12, 25)]


In [13]:
vendas_loja_dia = train.groupby(["store_nbr", "date"])["sales"].sum().reset_index()

dias_zerados = vendas_loja_dia[vendas_loja_dia["sales"] == 0]
print("Dias de loja sem nenhuma venda:", len(dias_zerados))

primeira_venda = vendas_loja_dia[vendas_loja_dia["sales"] > 0].groupby("store_nbr")["date"].min()
print("\nLojas que começaram a vender mais tarde:")
print(primeira_venda.sort_values(ascending=False).head(10))

Dias de loja sem nenhuma venda: 7330

Lojas que começaram a vender mais tarde:
store_nbr
52   2017-04-20
22   2015-10-09
42   2015-08-21
21   2015-07-24
29   2015-03-20
20   2015-02-13
53   2014-05-29
36   2013-05-09
39   2013-01-02
31   2013-01-02
Name: date, dtype: datetime64[us]


In [14]:
abertura = primeira_venda.rename("abertura")
zerados = dias_zerados.merge(abertura, on="store_nbr")

antes = zerados[zerados["date"] < zerados["abertura"]]
depois = zerados[zerados["date"] >= zerados["abertura"]]

print("Zerados antes da inauguração:", len(antes))
print("Zerados com a loja já aberta:", len(depois))
print("\nDatas mais comuns nos zerados com loja aberta:")
print(depois["date"].value_counts().head(10))

Zerados antes da inauguração: 6729
Zerados com a loja já aberta: 601

Datas mais comuns nos zerados com loja aberta:
date
2016-01-01    52
2017-01-01    52
2015-01-01    47
2014-01-01    45
2016-08-22     2
2016-08-23     2
2016-08-24     2
2016-08-25     2
2016-08-26     2
2016-08-27     2
Name: count, dtype: int64


- 7.330 dias de loja com venda zero:
  - 6.729 antes da inauguração (serão removidos no ETL);
  - ~200 em 1º de janeiro (lojas fechadas no Ano Novo);
  - o restante parece fechamento temporário de lojas.
- Limitação: inauguração = data da primeira venda, por isso o
  01/01/2013 foi contado como "antes da inauguração".